<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

# Questão 1

Utilize o dataset Iris disponível no scikit-learn.
Divida os dados em treino e teste utilizando divisão estratificada.

**Solução**:

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [ ]:
df = load_iris()
x = df.data
y = df.target

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,      
    stratify=y,         
    random_state=42     
)

print("Tamanho treino:", x_train.shape[0])
print("Tamanho teste:", x_test.shape[0])

# Questão 2

Treine um modelo utilizando `DecisionTreeClassifier`.

Depois calcule:

- acurácia no treino
- acurácia no teste

**Solução**:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
model = DecisionTreeClassifier(random_state=42)

model.fit(x_train, y_train)

# Previsões
y_train_pred = model.predict(x_train)
y_test_pred = model.predict(x_test)

# Acurácia
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Acurácia no treino: {train_accuracy*100}%")
print(f"Acurácia no teste: {test_accuracy*100:.4f}%")

# Questão 3

Utilize `plot_tree()` para visualizar a árvore treinada.

Responda:

1. Qual atributo aparece na raiz?
2. Qual é a profundidade da árvore?

**Solução**:

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(9,6))

plot_tree(
    model,
    feature_names=df.feature_names,
    class_names=df.target_names,
    filled=True
)

plt.show()

Qual atributo aparece na raiz?

R: Tamanho da pétala

Qual é a profundidade da árvore?

R: 5

# Questão 4

Treine dez árvores com:

- max_depth = 1
- max_depth = 2
- max_depth = 3
...
- max_depth = 9
- max_depth = None

Registre em uma tabela para cada árvore:

- acurácia no treino
- acurácia no teste
- profundidade da árvore
- número de folhas

**Solução**:

In [ ]:
import pandas as pd

In [ ]:
depths = [1,2,3,4,5,6,7,8,9,None]

results = []

for d in depths:
    
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(x_train, y_train)
    
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    depth = model.get_depth()
    leaves = model.get_n_leaves()
    
    results.append([d, round(train_acc,4), round(test_acc,4), depth, leaves])

df_results = pd.DataFrame(
    results,
    columns=[
        "max_depth",
        "acurácia_treino",
        "acurácia_teste",
        "profundidade",
        "n_folhas"
    ]
)

print(df_results)

Em qual profundidade começa o overfitting?

R: Com max_depth = 5, por que é quando o modelo começa a "decorar" os dados, notamos pela estagnação das acurácias. Aprendendo muito bem o treino mas com erros no teste.

Por que a árvore consegue 100% no treino quando max_depth=None?

R: Quando é None a árvore cresce infinitamente até que cada folha tenha apenas uma classe ou uma amostragem muito pequena. Parando em max_depth = 5 pois a partir disso não tinham mais decisões que faziam sentido

# Questão 5

Treine dois modelos:

- criterion = "gini"
- criterion = "entropy"

Compare:

- profundidade da árvore
- acurácia

**Solução**:

In [ ]:
criterions = ["gini", "entropy"]
results_criterion = []

for c in criterions:
    model = DecisionTreeClassifier(criterion=c, random_state=42)
    model.fit(x_train, y_train)
    
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    depth = model.get_depth()
    
    results_criterion.append([c, round(train_acc,4), round(test_acc,4), depth])

df_criterion = pd.DataFrame(
    results_criterion,
    columns=["criterion", "acurácia_treino", "acurácia_teste", "profundidade"]
)

print(df_criterion)

# Questão 6

Escolha um hiperparâmetro e investigue seu impacto.

Sugestões:

- max_depth
- min_samples_split
- min_samples_leaf
- criterion

Mostre resultados e interprete.
- melhor modelo encontrado
- acurácia
- parâmetros

**Solução**:

In [ ]:
leaf_values = [1, 2, 3, 4, 5, 10]

results_leaf = []

for leaf in leaf_values:
    model = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=42)
    model.fit(x_train, y_train)
    
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    depth = model.get_depth()
    leaves = model.get_n_leaves()
    
    results_leaf.append([leaf, round(train_acc,4), round(test_acc,4), depth, leaves])

df_leaf = pd.DataFrame(
    results_leaf,
    columns=["min_samples_leaf", "acurácia_treino", "acurácia_teste", "profundidade", "n_folhas"]
)

print(df_leaf)

No experimento com o hiperparâmetro min_samples_leaf, observamos que valores muito baixos (1 ou 2) geram overfitting, com acurácia de treino perfeita, mas menor no teste. Valores intermediários (3 ou 4) proporcionam melhor generalização, mantendo alta acurácia no teste (0.9667) e árvores mais rasas, simples e interpretáveis. Valores maiores (5 ou 10) começam a gerar underfitting, reduzindo a capacidade do modelo de capturar padrões. Portanto, o modelo mais equilibrado encontrado foi com min_samples_leaf = 3, equilibrando precisão e simplicidade da árvore.